In [1]:
import os
import pickle
import random

import faiss


# ============================================================
# Configuration
# ============================================================

BM25_FILE = "bm25_index.pkl"
CHUNK_STORE_FILE = "chunk_store.pkl"
FAISS_FILE = "dense_index.faiss"

# Number of chunk positions to inspect explicitly
POSITION_SAMPLES = [
    0,
    1,
    2,
    10,
    100,
    500,
    1000,
    -1,
]

# Number of random positions to inspect
RANDOM_SAMPLES = 10


# ============================================================
# Utility functions
# ============================================================

def load_pickle(path):
    print(f"\nLoading: {path}")

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "rb") as f:
        obj = pickle.load(f)

    print(f"  Type: {type(obj)}")

    return obj


def describe_object(name, obj):
    print(f"\n{'=' * 70}")
    print(f"{name}")
    print(f"{'=' * 70}")

    print(f"Type: {type(obj)}")

    if isinstance(obj, dict):
        print(f"Dictionary entries: {len(obj)}")

        print("\nFirst few keys:")
        for key in list(obj.keys())[:10]:
            print(f"  {repr(key)}")

    elif isinstance(obj, (list, tuple)):
        print(f"Length: {len(obj)}")

        if len(obj) > 0:
            print("\nFirst element:")
            print(f"  Type: {type(obj[0])}")
            print(f"  Value: {repr(obj[0])[:500]}")

    else:
        print("\nObject attributes:")
        attrs = [
            attr
            for attr in dir(obj)
            if not attr.startswith("_")
        ]

        for attr in attrs[:50]:
            try:
                value = getattr(obj, attr)

                if callable(value):
                    continue

                representation = repr(value)

                if len(representation) > 300:
                    representation = representation[:300] + "..."

                print(f"  {attr}: {representation}")

            except Exception:
                pass


# ============================================================
# Chunk-store inspection
# ============================================================

def get_chunk_count(chunk_store):
    """
    Attempt to determine how many chunks exist.

    Supports common structures:
        list
        tuple
        dict
    """

    if isinstance(chunk_store, (list, tuple)):
        return len(chunk_store)

    if isinstance(chunk_store, dict):
        return len(chunk_store)

    raise TypeError(
        "Unable to determine chunk count from chunk_store.pkl. "
        f"Unsupported type: {type(chunk_store)}"
    )


def get_chunk_at(chunk_store, position):
    """
    Retrieve a chunk using its positional index.

    For lists/tuples:
        chunk_store[position]

    For dictionaries:
        First attempt positional integer key.
        Otherwise use insertion-order position.
    """

    if isinstance(chunk_store, (list, tuple)):
        return chunk_store[position]

    if isinstance(chunk_store, dict):

        # Direct integer key
        if position in chunk_store:
            return chunk_store[position]

        # Fall back to insertion-order position
        keys = list(chunk_store.keys())

        return chunk_store[keys[position]]

    raise TypeError(
        f"Unsupported chunk store type: {type(chunk_store)}"
    )


def describe_chunk(chunk):
    """
    Produce a human-readable description of a chunk without
    assuming its internal structure.
    """

    if isinstance(chunk, str):
        return {
            "type": "str",
            "text": chunk,
        }

    if isinstance(chunk, dict):
        result = {
            "type": "dict",
            "keys": list(chunk.keys()),
        }

        # Look for common identifiers
        possible_id_keys = [
            "chunk_id",
            "id",
            "doc_id",
            "document_id",
            "index",
        ]

        possible_text_keys = [
            "text",
            "content",
            "chunk",
            "page_content",
        ]

        for key in possible_id_keys:
            if key in chunk:
                result["id_key"] = key
                result["id"] = chunk[key]
                break

        for key in possible_text_keys:
            if key in chunk:
                result["text_key"] = key
                result["text"] = chunk[key]
                break

        return result

    return {
        "type": type(chunk).__name__,
        "repr": repr(chunk)[:500],
    }


# ============================================================
# BM25 inspection
# ============================================================

def get_bm25_document_count(bm25):
    """
    Try to determine the number of documents represented
    by a BM25 object.

    Supports rank_bm25-style objects and common serialized
    structures.
    """

    # rank_bm25 normally exposes corpus_size
    if hasattr(bm25, "corpus_size"):
        return bm25.corpus_size

    # rank_bm25 also stores doc_freqs
    if hasattr(bm25, "doc_freqs"):
        return len(bm25.doc_freqs)

    # If a dictionary was serialized
    if isinstance(bm25, dict):

        possible_keys = [
            "corpus",
            "documents",
            "docs",
            "tokenized_corpus",
            "doc_freqs",
        ]

        for key in possible_keys:
            if key in bm25:
                value = bm25[key]

                try:
                    return len(value)
                except TypeError:
                    pass

    # If a list/tuple was serialized directly
    if isinstance(bm25, (list, tuple)):
        return len(bm25)

    return None


def get_bm25_corpus(bm25):
    """
    Attempt to recover the original tokenized corpus from
    common BM25 representations.

    This is optional. Alignment checking can still proceed
    without it.
    """

    # Some custom implementations may store corpus directly
    if hasattr(bm25, "corpus"):
        return bm25.corpus

    if isinstance(bm25, dict):
        for key in [
            "corpus",
            "documents",
            "docs",
            "tokenized_corpus",
        ]:
            if key in bm25:
                return bm25[key]

    return None


# ============================================================
# Main checks
# ============================================================

def main():

    print("\n")
    print("=" * 70)
    print("RAG INDEX ALIGNMENT CHECK")
    print("=" * 70)

    # --------------------------------------------------------
    # Check that files exist
    # --------------------------------------------------------

    print("\nChecking files...")

    files = [
        BM25_FILE,
        CHUNK_STORE_FILE,
        FAISS_FILE,
    ]

    all_files_exist = True

    for path in files:
        exists = os.path.exists(path)

        print(
            f"  {'PASS' if exists else 'FAIL'}  "
            f"{path}"
        )

        if not exists:
            all_files_exist = False

    if not all_files_exist:
        print("\nCannot continue because one or more files are missing.")
        return

    # --------------------------------------------------------
    # Load artifacts
    # --------------------------------------------------------

    try:
        bm25 = load_pickle(BM25_FILE)
    except Exception as e:
        print(f"\nERROR loading BM25 index:")
        print(f"  {e}")
        return

    try:
        chunk_store = load_pickle(CHUNK_STORE_FILE)
    except Exception as e:
        print(f"\nERROR loading chunk store:")
        print(f"  {e}")
        return

    try:
        print(f"\nLoading: {FAISS_FILE}")

        dense_index = faiss.read_index(FAISS_FILE)

        print(f"  Type: {type(dense_index)}")
        print(f"  Vectors: {dense_index.ntotal}")
        print(f"  Dimensions: {dense_index.d}")

    except Exception as e:
        print(f"\nERROR loading FAISS index:")
        print(f"  {e}")
        return

    # --------------------------------------------------------
    # Describe structures
    # --------------------------------------------------------

    describe_object("BM25 INDEX STRUCTURE", bm25)
    describe_object("CHUNK STORE STRUCTURE", chunk_store)

    # --------------------------------------------------------
    # Determine sizes
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("SIZE / COUNT CHECK")
    print("=" * 70)

    try:
        chunk_count = get_chunk_count(chunk_store)
    except Exception as e:
        print(f"\nCould not determine chunk count:")
        print(f"  {e}")
        chunk_count = None

    bm25_count = get_bm25_document_count(bm25)
    faiss_count = dense_index.ntotal

    print(f"\nChunk store:")
    print(f"  {chunk_count}")

    print(f"\nBM25:")
    print(f"  {bm25_count}")

    print(f"\nFAISS:")
    print(f"  {faiss_count}")

    if (
        chunk_count is not None
        and bm25_count is not None
        and faiss_count is not None
    ):

        if chunk_count == bm25_count == faiss_count:
            print("\nCOUNT CHECK: PASS")
            print("All three structures contain the same number of entries.")

        else:
            print("\nCOUNT CHECK: FAIL")

            print(
                "\nThis is a serious problem. "
                "The three artifacts cannot safely be assumed to be aligned."
            )

    # --------------------------------------------------------
    # Inspect chunk positions
    # --------------------------------------------------------

    if chunk_count is None or chunk_count == 0:
        print("\nNo chunks available for positional inspection.")
        return

    print("\n")
    print("=" * 70)
    print("CHUNK POSITION INSPECTION")
    print("=" * 70)

    positions = set()

    for position in POSITION_SAMPLES:

        if position < 0:
            actual_position = chunk_count + position
        else:
            actual_position = position

        if 0 <= actual_position < chunk_count:
            positions.add(actual_position)

    random.seed(42)

    while len(positions) < min(
        len(POSITION_SAMPLES) + RANDOM_SAMPLES,
        chunk_count,
    ):
        positions.add(random.randrange(chunk_count))

    positions = sorted(positions)

    for position in positions:

        try:
            chunk = get_chunk_at(
                chunk_store,
                position,
            )

            description = describe_chunk(chunk)

            print(f"\nPosition {position}")
            print(f"  Type: {description['type']}")

            if "id" in description:
                print(
                    f"  Chunk ID: {repr(description['id'])}"
                )

            if "text" in description:

                text = str(description["text"])

                text_preview = (
                    text[:250]
                    .replace("\n", " ")
                )

                print(
                    f"  Text: {text_preview}"
                    f"{'...' if len(text) > 250 else ''}"
                )

        except Exception as e:

            print(
                f"\nPosition {position}: FAIL"
            )

            print(
                f"  Could not retrieve chunk: {e}"
            )

    # --------------------------------------------------------
    # Inspect BM25 corpus if available
    # --------------------------------------------------------

    bm25_corpus = get_bm25_corpus(bm25)

    print("\n")
    print("=" * 70)
    print("BM25 CORPUS CHECK")
    print("=" * 70)

    if bm25_corpus is None:

        print(
            "\nBM25 corpus is not directly accessible "
            "from the serialized object."
        )

        print(
            "This does NOT mean BM25 is broken."
        )

        print(
            "It simply means we cannot perform a direct "
            "document-by-document identity comparison."
        )

    else:

        print(
            f"\nRecovered BM25 corpus length: "
            f"{len(bm25_corpus)}"
        )

        if chunk_count == len(bm25_corpus):

            print("BM25 corpus count: PASS")

        else:

            print("BM25 corpus count: FAIL")

    # --------------------------------------------------------
    # FAISS basic sanity
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("FAISS SANITY CHECK")
    print("=" * 70)

    print(f"\nFAISS vectors: {dense_index.ntotal}")
    print(f"Vector dimension: {dense_index.d}")

    if dense_index.ntotal == chunk_count:

        print("\nFAISS ↔ chunk count: PASS")

    else:

        print("\nFAISS ↔ chunk count: FAIL")

    # --------------------------------------------------------
    # Final verdict
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("PRELIMINARY VERDICT")
    print("=" * 70)

    if (
        chunk_count is not None
        and bm25_count is not None
        and faiss_count is not None
        and chunk_count == bm25_count == faiss_count
    ):

        print(
            "\nPASS — All three artifacts have matching counts."
        )

        print(
            "\nThis establishes positional compatibility, "
            "but not yet semantic identity."
        )

        print(
            "The next step is to test whether the same chunk "
            "position actually represents the same document "
            "across FAISS, BM25, and chunk_store."
        )

    else:

        print(
            "\nFAIL — The artifacts do not currently have "
            "matching dimensions/counts."
        )

        print(
            "Do NOT build the retrieval tool until this is resolved."
        )

    print("\n")


if __name__ == "__main__":
    main()



RAG INDEX ALIGNMENT CHECK

Checking files...
  PASS  bm25_index.pkl
  PASS  chunk_store.pkl
  PASS  dense_index.faiss

Loading: bm25_index.pkl
  Type: <class 'rank_bm25.BM25Okapi'>

Loading: chunk_store.pkl
  Type: <class 'list'>

Loading: dense_index.faiss
  Type: <class 'faiss.swigfaiss.IndexFlatIP'>
  Vectors: 270296
  Dimensions: 768

BM25 INDEX STRUCTURE
Type: <class 'rank_bm25.BM25Okapi'>

Object attributes:
  average_idf: 9.97006479698973
  avgdl: 381.6079668215586
  b: 0.75
  corpus_size: 270296
  doc_freqs: [{'senjō': 1, 'no': 1, 'valkyria': 9, '3': 2, ':': 3, '<': 3, 'unk': 3, '>': 3, 'chronicles': 6, '(': 1, 'japanese': 2, '戦場のヴァルキュリア3': 1, ',': 15, 'lit': 1, '.': 14, 'of': 9, 'the': 22, 'battlefield': 1, ')': 1, 'commonly': 1, 'referred': 1, 'to': 4, 'as': 3, 'iii': 2, 'outside': 1, 'japan': 3, 'is'...
  doc_len: [329, 615, 634, 530, 312, 357, 439, 66, 580, 191, 221, 977, 968, 737, 284, 165, 456, 252, 164, 211, 165, 295, 251, 140, 239, 218, 369, 104, 413, 21, 176, 134, 37